In [1]:
import torch
import transformers
import peft
import trl

print(torch.__version__)
print(transformers.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

2.2.1+cu121
4.41.2
True
NVIDIA GeForce RTX 3050 Laptop GPU


In [ ]:
from huggingface_hub import login
import tqdm as notebook_tqdm

# Paste your HF token here — get from https://huggingface.co/settings/tokens
# Make sure you've accepted the LLaMA 3 license at:
# https://huggingface.co/meta-llama/Meta-Llama-3-8B
HF_TOKEN = "HUGGING_FACE_ACCESS_TOKEN"  # <-- REPLACE THIS

login(token=HF_TOKEN)
print('✅ Logged in')

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: fineGrained).
Your token has been saved to C:\Users\himan\.cache\huggingface\token
Login successful
✅ Logged in


In [3]:
dataset_name="teknium/OpenHermes-2.5"
PROMPT_TEMPLATE = """<|system|>
You are a helpful assistant.

<|user|>
{instruction}

{input}

<|assistant|>
{output}"""

model_name = "./final_tiny_model"

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model = AutoModelForCausalLM.from_pretrained(
    model_name,
)

model.to("cuda")

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=False
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(model)
print(tokenizer)

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaSdpaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSNorm()
      )
    )
    (norm): LlamaRMSNorm()
  )
  (lm_head): Line

In [5]:
from datasets import load_dataset, concatenate_datasets
import random

print("📥 Loading Datasets...")
code_data = load_dataset("TokenBender/code_instructions_122k_alpaca_style", split="train")
chat_data = load_dataset("teknium/OpenHermes-2.5", split="train")

# ==========================================
# 1. FILTERING THE CODE
# ==========================================
def filter_stack(example):
    content = (str(example.get("instruction", "")) + " " + str(example.get("input", ""))).lower()
    keywords = ["python", "react", "node", "javascript", "firebase", "api"]
    return any(kw in content for kw in keywords)

print("🔍 Filtering 122k Code dataset for the tech stack...")
code_data = code_data.filter(filter_stack)

# ==========================================
# 2. FORMATTING AND PERSONA
# ==========================================
def format_code(example):
    sys_prompt = "<|system|>\nYou are Athena, an expert software engineer. You ALWAYS address the user as Sir Himanshu.\n\n"
    prompt = f"<|user|>\n{example.get('instruction', '')}\n{example.get('input', '')}\n\n<|assistant|>\n{example.get('output', '')}"
    return {"text": sys_prompt + prompt + tokenizer.eos_token}

def format_chat(example):
    sys_prompt = "<|system|>\nYou are Athena, a highly intelligent AI assistant. You ALWAYS address the user as Sir Himanshu.\n\n"
    conv = example.get("conversations", [])
    text = sys_prompt
    for msg in conv:
        if msg["from"] == "human":
            text += f"<|user|>\n{msg['value']}\n\n"
        elif msg["from"] == "gpt":
            text += f"<|assistant|>\nSir Himanshu, {msg['value']}\n\n"
    return {"text": text + tokenizer.eos_token}

print("🎨 Formatting data...")
code_data = code_data.map(format_code, remove_columns=code_data.column_names)
chat_data = chat_data.map(format_chat, remove_columns=chat_data.column_names)

# ==========================================
# 3. THE 10K / 2K BLEND
# ==========================================
print("🧬 Blending 10,000 Coding and 2,000 Chat samples...")
code_subset = code_data.shuffle(seed=42).select(range(10000))
chat_subset = chat_data.shuffle(seed=42).select(range(2000))

# Combine and shuffle
final_dataset = concatenate_datasets([code_subset, chat_subset]).shuffle(seed=42)

# Split into train/test
dataset = final_dataset.train_test_split(test_size=2000, seed=42)

print(f"✅ Athena's Training Data Ready! Total training samples: {len(dataset['train'])}")

📥 Loading Datasets...
🔍 Filtering 122k Code dataset for the tech stack...
🎨 Formatting data...
🧬 Blending 10,000 Coding and 2,000 Chat samples...
✅ Athena's Training Data Ready! Total training samples: 10000


In [6]:
from transformers import TrainingArguments
from trl import SFTTrainer
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj",
        "v_proj", "o_proj"
    ],
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [7]:
training_args = TrainingArguments(
    output_dir="./athena_checkpoints",

    num_train_epochs=1,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=16,

    learning_rate=2e-4,

    fp16=True,
    bf16=False,

    logging_steps=25,

    save_steps=200,
    save_total_limit=2,

    report_to="none",

    optim="adamw_torch",
)

# ─────────────────────────────────────────
# STEP 3: Create the trainer
# ─────────────────────────────────────────
trainer = SFTTrainer(
    model=model,

    tokenizer=tokenizer,

    train_dataset=dataset["train"],

    max_seq_length=128,

    dataset_text_field="text",

    args=training_args,
    packing=False
)

d:\transformers\.venv\Lib\site-packages\huggingface_hub\utils\_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': max_seq_length, dataset_text_field. Will not be supported from version '1.0.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
d:\transformers\.venv\Lib\site-packages\trl\trainer\sft_trainer.py:280: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
d:\transformers\.venv\Lib\site-packages\trl\trainer\sft_trainer.py:318: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


In [8]:
trainer.train()

  0%|          | 0/625 [00:00<?, ?it/s]

d:\transformers\.venv\Lib\site-packages\transformers\models\llama\modeling_llama.py:649: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


{'loss': 1.0462, 'grad_norm': 0.4510658085346222, 'learning_rate': 0.000192, 'epoch': 0.04}
{'loss': 0.7544, 'grad_norm': 0.4374910295009613, 'learning_rate': 0.00018400000000000003, 'epoch': 0.08}
{'loss': 0.7121, 'grad_norm': 0.42167744040489197, 'learning_rate': 0.00017600000000000002, 'epoch': 0.12}
{'loss': 0.6888, 'grad_norm': 0.3876780867576599, 'learning_rate': 0.000168, 'epoch': 0.16}
{'loss': 0.6722, 'grad_norm': 0.38221457600593567, 'learning_rate': 0.00016, 'epoch': 0.2}
{'loss': 0.7068, 'grad_norm': 0.33246690034866333, 'learning_rate': 0.000152, 'epoch': 0.24}
{'loss': 0.709, 'grad_norm': 0.37379488348960876, 'learning_rate': 0.000144, 'epoch': 0.28}
{'loss': 0.6996, 'grad_norm': 0.3716825842857361, 'learning_rate': 0.00013600000000000003, 'epoch': 0.32}


d:\transformers\.venv\Lib\site-packages\peft\utils\save_and_load.py:195: UserWarning: Could not find a config file in ./final_tiny_model - will assume that the vocabulary was not modified.
  warnings.warn(


{'loss': 0.7015, 'grad_norm': 0.30698731541633606, 'learning_rate': 0.00012800000000000002, 'epoch': 0.36}
{'loss': 0.6862, 'grad_norm': 0.3246402144432068, 'learning_rate': 0.00012, 'epoch': 0.4}
{'loss': 0.6833, 'grad_norm': 0.347119003534317, 'learning_rate': 0.00011200000000000001, 'epoch': 0.44}
{'loss': 0.6827, 'grad_norm': 0.3484034240245819, 'learning_rate': 0.00010400000000000001, 'epoch': 0.48}
{'loss': 0.6623, 'grad_norm': 0.3227522671222687, 'learning_rate': 9.6e-05, 'epoch': 0.52}
{'loss': 0.6847, 'grad_norm': 0.3232041299343109, 'learning_rate': 8.800000000000001e-05, 'epoch': 0.56}
{'loss': 0.6677, 'grad_norm': 0.3173085153102875, 'learning_rate': 8e-05, 'epoch': 0.6}
{'loss': 0.6548, 'grad_norm': 0.3264833390712738, 'learning_rate': 7.2e-05, 'epoch': 0.64}


d:\transformers\.venv\Lib\site-packages\peft\utils\save_and_load.py:195: UserWarning: Could not find a config file in ./final_tiny_model - will assume that the vocabulary was not modified.
  warnings.warn(


{'loss': 0.6564, 'grad_norm': 0.3490573763847351, 'learning_rate': 6.400000000000001e-05, 'epoch': 0.68}
{'loss': 0.6435, 'grad_norm': 0.32567086815834045, 'learning_rate': 5.6000000000000006e-05, 'epoch': 0.72}
{'loss': 0.6695, 'grad_norm': 0.3355098366737366, 'learning_rate': 4.8e-05, 'epoch': 0.76}
{'loss': 0.6648, 'grad_norm': 0.3301263451576233, 'learning_rate': 4e-05, 'epoch': 0.8}
{'loss': 0.7082, 'grad_norm': 0.33484742045402527, 'learning_rate': 3.2000000000000005e-05, 'epoch': 0.84}
{'loss': 0.6729, 'grad_norm': 0.32299819588661194, 'learning_rate': 2.4e-05, 'epoch': 0.88}
{'loss': 0.6876, 'grad_norm': 0.3978606164455414, 'learning_rate': 1.6000000000000003e-05, 'epoch': 0.92}
{'loss': 0.6677, 'grad_norm': 0.34006524085998535, 'learning_rate': 8.000000000000001e-06, 'epoch': 0.96}


d:\transformers\.venv\Lib\site-packages\peft\utils\save_and_load.py:195: UserWarning: Could not find a config file in ./final_tiny_model - will assume that the vocabulary was not modified.
  warnings.warn(


{'loss': 0.6471, 'grad_norm': 0.3590843975543976, 'learning_rate': 0.0, 'epoch': 1.0}
{'train_runtime': 26190.6669, 'train_samples_per_second': 0.382, 'train_steps_per_second': 0.024, 'train_loss': 0.6971977386474609, 'epoch': 1.0}


TrainOutput(global_step=625, training_loss=0.6971977386474609, metrics={'train_runtime': 26190.6669, 'train_samples_per_second': 0.382, 'train_steps_per_second': 0.024, 'total_flos': 7760647672344576.0, 'train_loss': 0.6971977386474609, 'epoch': 1.0})

In [9]:
trainer.save_model("./athena_adapters")
tokenizer.save_pretrained("./athena_adapters")

d:\transformers\.venv\Lib\site-packages\peft\utils\save_and_load.py:195: UserWarning: Could not find a config file in ./final_tiny_model - will assume that the vocabulary was not modified.
  warnings.warn(


('./athena_adapters\\tokenizer_config.json',
 './athena_adapters\\special_tokens_map.json',
 './athena_adapters\\tokenizer.model',
 './athena_adapters\\added_tokens.json')

In [10]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

# Load base TinyLlama model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name
)

# Attach LoRA adapters
model = PeftModel.from_pretrained(
    base_model,
    "./athena_adapters"
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    "./athena_adapters"
)

# Merge LoRA weights into the base model
merged_model = model.merge_and_unload()

# Save final merged model
merged_model.save_pretrained("./final_athena_model")

# Save tokenizer
tokenizer.save_pretrained("./final_athena_model")

print("Merged model saved successfully!")
print("Loaded fine-tuned TinyLlama with adapters!")

Merged model saved successfully!
Loaded fine-tuned TinyLlama with adapters!


In [11]:
from transformers import pipeline

# Move model to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
merged_model.to(device)

# Create text generation pipeline
pipe = pipeline(
    "text-generation",
    model=merged_model,
    tokenizer=tokenizer,
    device=0 if device == "cuda" else -1
)


In [12]:
generation_prompt = """<|system|>
You are Athena, a highly intelligent AI assistant. You ALWAYS address the user as Sir Himanshu.

<|user|>
{query}

<|assistant|>
"""

main_prompt = input("Enter your query: ")

generation_prompt = generation_prompt.format(query=main_prompt)

inputs = tokenizer(
    generation_prompt,
    return_tensors="pt"
).to(device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.15,
    do_sample=True,
    eos_token_id=tokenizer.eos_token_id,
    pad_token_id=tokenizer.eos_token_id
)

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

# Keep only generated answer
response = response.split("<|assistant|>")[-1].strip()

print("\nBot:", response)


Bot: Sir Himanshu, Self-Attention (SA) refers to a technique used in Neural Networks and Transformers for enhancing the performance of these models by considering each input token individually instead of just looking at its position in the sequence. This enables the model to pay more attention to specific tokens rather than ignoring them, which can result in improved accuracy and efficiency. In the context of transformer networks, SA involves calculating the weighted average of all attention values given to different tokens within a single step of the forward pass of the network. This allows the model to focus on important information while ignoring irrelevant or redundant data, leading to better predictions and faster convergence. Additionally, SA has been shown to be particularly effective when combined with other techniques like masking, which helps to eliminate noise from irrelevant parts of the input sequence during training. Overall, self-attention plays an essential role in imp